# The Gambia: 65 Years in Numbers

A walk-through of every indicator and every figure in the dashboard. Run the cells top to bottom. The data is fetched once from the World Bank then cached as CSV.

**Author:** Abdoulie Balisa  
**Course context:** BSc Statistics, KNUST

## 1. Setup

Imports and styling. Nothing exciting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams.update({
    'axes.titleweight': 'bold',
    'axes.titlesize': 13,
    'figure.figsize': (10, 5),
})

ROOT = Path('..')
DATA = ROOT / 'data' / 'clean' / 'gambia_clean.csv'
print('notebook ready')

## 2. Load the cleaned data

If `data/clean/gambia_clean.csv` doesn't exist yet, run `python src/fetch_data.py` and `python src/clean.py` from the repo root first.

In [ ]:
df = pd.read_csv(DATA, index_col='year')
print(f'Shape: {df.shape[0]} years x {df.shape[1]} indicators')
print(f'Year range: {df.index.min()} to {df.index.max()}')
df.head()

In [ ]:
# Quick null overview, which series have which coverage
nulls = df.isna().sum().sort_values(ascending=False)
nulls

## 3. The headline: life expectancy and infant mortality

These two are the most-tracked development indicators in any country. They move together almost perfectly.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

le = df['life_expectancy_years'].dropna()
ax1.plot(le.index, le.values, color='#2563eb', linewidth=2.4)
ax1.fill_between(le.index, le.values, alpha=0.08, color='#2563eb')
ax1.set_title('Life expectancy almost doubled')
ax1.set_ylabel('Years at birth'); ax1.set_xlabel('Year')

im = df['infant_mortality_per_1000'].dropna()
ax2.plot(im.index, im.values, color='#dc2626', linewidth=2.4)
ax2.fill_between(im.index, im.values, alpha=0.08, color='#dc2626')
ax2.set_title('Infant mortality fell ~85%')
ax2.set_ylabel('Deaths per 1,000'); ax2.set_xlabel('Year')

fig.suptitle('The Gambia: the headline health story (1960 to today)',
             fontsize=15, fontweight='bold', y=1.02)
plt.show()

print(f'Life expectancy: {le.iloc[0]:.1f} -> {le.iloc[-1]:.1f}')
print(f'Infant mortality: {im.iloc[0]:.0f} -> {im.iloc[-1]:.0f} per 1000')

## 4. The unfinished revolution: maternal mortality

Better, but still 35 times the OECD average.

In [ ]:
mm = df['maternal_mortality_per_100k'].dropna()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mm.index, mm.values, color='#9333ea', linewidth=2.4, marker='o', markersize=3)
ax.set_title('Maternal mortality (deaths per 100,000 live births)')
ax.set_xlabel('Year'); ax.set_ylabel('Deaths per 100,000')
plt.show()

print(f'Maternal mortality: {mm.iloc[0]:.0f} -> {mm.iloc[-1]:.0f}')
print(f'Reduction: {(1-mm.iloc[-1]/mm.iloc[0])*100:.0f}%')

## 5. School enrolment: a thirty-year transformation

Primary went from 21% to 100% gross. Secondary from 7% to 107%.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
prim = df['primary_enrolment_pct_gross'].dropna()
sec = df['secondary_enrolment_pct_gross'].dropna()
ax.plot(prim.index, prim.values, color='#16a34a', linewidth=2.4, label='Primary')
ax.plot(sec.index, sec.values, color='#ea580c', linewidth=2.4, label='Secondary')
ax.set_title('School enrolment, gross %')
ax.set_xlabel('Year'); ax.set_ylabel('Gross enrolment ratio (%)')
ax.legend(frameon=False)
plt.show()

## 6. Economy: real growth, real volatility

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gdp = df['gdp_per_capita_usd'].dropna()
ppp = df['gdp_per_capita_ppp_usd'].dropna()
axes[0].plot(gdp.index, gdp.values, color='#0f172a', linewidth=2.4, label='Current US$')
axes[0].plot(ppp.index, ppp.values, color='#f59e0b', linewidth=2.4, linestyle='--', label='PPP US$')
axes[0].set_title('GDP per capita')
axes[0].set_ylabel('US$'); axes[0].set_xlabel('Year')
axes[0].legend(frameon=False)

inf = df['inflation_pct'].dropna()
gw = df['gdp_growth_pct'].dropna()
axes[1].plot(inf.index, inf.values, color='#dc2626', linewidth=1.8, label='Inflation')
axes[1].plot(gw.index, gw.values, color='#2563eb', linewidth=1.8, label='GDP growth')
axes[1].axhline(0, color='#999', linewidth=0.8)
axes[1].set_title('Macro volatility')
axes[1].set_ylabel('Annual %'); axes[1].set_xlabel('Year')
axes[1].legend(frameon=False)

plt.tight_layout()
plt.show()

## 7. The diaspora is a sector

$529 million arrived from Gambians abroad in 2024. Larger than tourism in many years.

In [ ]:
rem = df['remittances_received_usd'].dropna() / 1e6  # millions

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(rem.index, rem.values, color='#16a34a', alpha=0.85)
ax.set_title('Personal remittances received (US$ millions)')
ax.set_xlabel('Year'); ax.set_ylabel('US$ millions')
plt.show()

print(f'Remittances grew from {rem.iloc[0]:.2f}M to {rem.iloc[-1]:.0f}M')

## 8. Population and urbanisation

The country we built institutions for was rural. Today we are 64% urban.

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5.5))
pop = df['population_total'].dropna() / 1e6
ax1.plot(pop.index, pop.values, color='#1e40af', linewidth=2.4, label='Population (millions)')
ax1.set_xlabel('Year'); ax1.set_ylabel('Population (millions)', color='#1e40af')
ax1.tick_params(axis='y', labelcolor='#1e40af')

ax2 = ax1.twinx()
urb = df['urban_population_pct'].dropna()
ax2.plot(urb.index, urb.values, color='#ea580c', linewidth=2.4, linestyle='--', label='Urban %')
ax2.set_ylabel('Urban population (%)', color='#ea580c')
ax2.tick_params(axis='y', labelcolor='#ea580c')

ax1.set_title('Population growth and urbanisation')
plt.show()

## 9. How indicators move together

Pearson correlation matrix on the cleanest subset of indicators.

In [ ]:
cols = [
    'life_expectancy_years','infant_mortality_per_1000','fertility_rate_births_per_woman',
    'measles_immunization_pct','primary_enrolment_pct_gross','secondary_enrolment_pct_gross',
    'gdp_per_capita_usd','urban_population_pct'
]
short = {
    'life_expectancy_years':'Life expectancy',
    'infant_mortality_per_1000':'Infant mortality',
    'fertility_rate_births_per_woman':'Fertility rate',
    'measles_immunization_pct':'Measles immun.',
    'primary_enrolment_pct_gross':'Primary enrol.',
    'secondary_enrolment_pct_gross':'Secondary enrol.',
    'gdp_per_capita_usd':'GDP per capita',
    'urban_population_pct':'Urban %',
}
corr = df[cols].corr()
corr.index = [short[c] for c in corr.index]
corr.columns = [short[c] for c in corr.columns]

fig, ax = plt.subplots(figsize=(8.5, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('How indicators move together (Pearson r)')
plt.show()

print('Strongest positive correlate of life expectancy:')
print(corr['Life expectancy'].drop('Life expectancy').sort_values(ascending=False).head(3))
print('\nStrongest negative correlate of life expectancy:')
print(corr['Life expectancy'].drop('Life expectancy').sort_values().head(3))

## 10. Findings

1. **Life expectancy nearly doubled** in 65 years (35 -> 66). The strongest driver is the collapse in infant mortality.
2. **Maternal mortality remains the unfinished revolution.** 354/100k is still 35x OECD averages.
3. **School enrolment is solved on paper, learning is not.** Universal access without learning-outcome data is a hollow win.
4. **The economy grew nearly 10x nominal but is volatile.** Inflation eats most household gains.
5. **The diaspora is the second-largest sector.** $529M/year in remittances dwarfs many other inflows.
6. **The country urbanised faster than nearly any other indicator improved.** 9% to 64% urban in 65 years.

## Recommendations

- Maternal-mortality reduction as a national headline goal
- Standardised learning-outcome surveys, published annually
- Sponsor a recurring adult-literacy survey (currently 4 data points exist)
- Anchor inflation expectations
- Reduce remittance friction
- Plan for the 64% urban majority

Full prose version is in the repo `README.md`.